# AdaBoost-FKD example

In this notebook, we present how to execute the AdaBoost-FKD algoritm, as well as the local version without the federated process, for comparison purposes.
It is assumed that with this example, the rest of experiments could be also addressed.

In [1]:
# Import data from flextrees
# TO-DO: Habria que poner algun ejemplo que no use dataset de flex?
from flextrees.datasets.tabular_datasets import adult
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Import AdaBoostFKD, and also the Federated Random Forest, for comparison
from models.AdaBoostFKD import AdaBoostFKD
from models.FRF import FRF_eval

from sklearn.datasets import load_breast_cancer

In [2]:
# Fix the random numbers seed
seed = 0

Load the dataset.

Note that the whole dataset (both train and test) is loaded and joined together. Later, the whole data is distributed among the clients to simulate the federated scenario.

In [3]:
train_data, test_data = adult(ret_feature_names=False, categorical=False)
X_data,y_data = train_data.to_numpy()
X_test,y_test = test_data.to_numpy()
X_data = np.concatenate((X_data,X_test))
y_data = np.concatenate((y_data,y_test))

#Alternatively, you can directly use any numpy dataset of your own:

#bcancer = load_breast_cancer()
#X_data = bcancer.data
#y_data = bcancer.target

In [4]:
# Separate public data (a small portion, for example, 5%)
# It is considered to be unlabeled, so we do not store the targets of the public data
data, public_data, targets, _ = train_test_split(X_data, y_data, test_size=0.05, random_state=seed)

# Get global test set (i.e., 10%), and the rest of training data that will be later distributed among clients
X_train, X_test, y_train, y_test = train_test_split(data,targets,test_size=0.1,random_state=seed)

Run AdaBoost-FKD

In [5]:
# When creating the model the data partition according to the chosen data distribution is made. 
fl_model = AdaBoostFKD(X_train, y_train, public_data, 
                         n_clients=10, T=10,
                         data_distribution='niid_quantity_skew', distribution_param=0.5,
                         public_data_prediction='weighted_majority_voting', 
                         server_alpha_weight_adj='common_weighted',
                         prediction_weights='only_server',soft_predictions=True, temperature=3,
                         client_weight_adj='common',    
                         random_state=seed,
                         server_classifier=DecisionTreeClassifier, server_classifier_params={'random_state':seed, 'max_depth':4, 'max_leaf_nodes':None},
                         clients_classifier=DecisionTreeClassifier, clients_classifier_params={'random_state':seed, 'max_depth':4, 'max_leaf_nodes':None})

# Store data distrib for subsequent models
train_dict = fl_model.train_clients_data.copy()
test_dict = fl_model.test_clients_data.copy()

# Takes the columns of weights of AdaBoost out
for key,(train,labeltr) in train_dict.items():
    train_dict[key] = (train[:,:-1],labeltr)
    test,labelte = test_dict[key]
    test_dict[key] = (test[:,:-1],labelte)

In [6]:
#Train the model
fl_model.fitmodel()

In [7]:
# Evaluate how the federated model works in local tests and global test (in average among clients) 
fl_acc_global, fl_f1_global, fl_acc_local, fl_f1_local = fl_model.overall_score(X_test, y_test)

print(f'AdaBoost-FKD accuracy (global test): {fl_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {fl_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {fl_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {fl_f1_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.8484
AdaBoost-FKD f1-score (global test): 0.8548
AdaBoost-FKD accuracy (local test): 0.8591
AdaBoost-FKD f1-score (local test): 0.8688


Run local AdaBoost models at each client

In [8]:
# If we want to compare it to the local models (without federated process), we first need to train local models with same data
fl_model.fit_local_clients_models()

In [9]:
# Then we can get a Dataframe with all the results for each client
localAB_acc_scores = fl_model.overall_acc_score(X_test, y_test)
localAB_acc_scores

,data_distrib,FL_acc_own_data,FL_acc_global_data,local_acc_own_data,local_acc_global_data,local_difference,global_difference
0,4011.0,0.860419,0.848416,0.856431,0.845184,0.003988,0.003232
1,20.0,1.000000,0.848416,0.833333,0.779250,0.166667,0.069166
2,2464.0,0.862013,0.848416,0.832792,0.847123,0.029221,0.001293
3,4073.0,0.865554,0.848416,0.861629,0.843891,0.003925,0.004525
4,696.0,0.880000,0.848416,0.851429,0.837104,0.028571,0.011312
5,760.0,0.842932,0.848416,0.816754,0.832902,0.026178,0.015514
6,170.0,0.697674,0.848416,0.767442,0.814480,-0.069767,0.033937
7,680.0,0.877193,0.848416,0.830409,0.821267,0.046784,0.027149
8,9073.0,0.844866,0.848416,0.833848,0.840336,0.011018,0.008080
9,316.0,0.860759,0.848416,0.835443,0.834195,0.025316,0.014221


In [10]:
localAB_f1_scores = fl_model.overall_F1_score(X_test,y_test)
localAB_f1_scores

,data_distrib,FL_wf1_own_data,FL_wf1_global_data,local_wf1_own_data,local_wf1_global_data,local_difference_w,global_difference_w
0,4011.0,0.869809,0.854784,0.869771,0.856981,0.000038,-0.002197
1,20.0,1.000000,0.854784,0.909091,0.821125,0.090909,0.033659
2,2464.0,0.870269,0.854784,0.842797,0.856026,0.027472,-0.001243
3,4073.0,0.870511,0.854784,0.869863,0.854820,0.000648,-0.000036
4,696.0,0.885236,0.854784,0.854042,0.842573,0.031194,0.012211
5,760.0,0.851914,0.854784,0.820601,0.832255,0.031313,0.022529
6,170.0,0.736927,0.854784,0.773363,0.820150,-0.036436,0.034633
7,680.0,0.882285,0.854784,0.825466,0.816924,0.056819,0.037859
8,9073.0,0.851414,0.854784,0.843062,0.848037,0.008352,0.006746
9,316.0,0.870065,0.854784,0.859496,0.844447,0.010569,0.010337


In [11]:
# Evaluate how the local models works in local tests and global test (in average among them) 
localAB_acc_global = localAB_acc_scores['local_acc_global_data'].mean()
localAB_f1_global = localAB_f1_scores['local_wf1_global_data'].mean()
localAB_acc_local = localAB_acc_scores['local_acc_own_data'].mean()
localAB_f1_local = localAB_f1_scores['local_wf1_own_data'].mean()

print(f'AdaBoost-FKD accuracy (global test): {localAB_acc_global:.4f}')
print(f'AdaBoost-FKD f1-score (global test): {localAB_f1_global:.4f}')
print(f'AdaBoost-FKD accuracy (local test): {localAB_acc_local:.4f}')
print(f'AdaBoost-FKD f1-score (local test): {localAB_f1_local:.4f}')

AdaBoost-FKD accuracy (global test): 0.8296
AdaBoost-FKD f1-score (global test): 0.8393
AdaBoost-FKD accuracy (local test): 0.8320
AdaBoost-FKD f1-score (local test): 0.8468


Compare to state-of-the-art methods

In [12]:
# Now we can use it to compare it to other state of the art algorithms such as FRF with 100 estimators
FRF_acc_global, FRF_f1_global, FRF_acc_local, FRF_f1_local = FRF_eval(train_dict, test_dict, X_test, y_test, hyperparameters='theirs')

In [13]:
print(f'FRF model accuracy (global test): {FRF_acc_global:.4f}')
print(f'FRF model f1-score (global test): {FRF_f1_global:.4f}')
print(f'FRF model accuracy (local test): {FRF_acc_local:.4f}')
print(f'FRF model f1-score (local test): {FRF_f1_local:.4f}')

FRF model accuracy (global test): 0.8387
FRF model f1-score (global test): 0.8201
FRF model accuracy (local test): 0.8270
FRF model f1-score (local test): 0.7991
